# Distorted Visual Sequence Pattern Recognition
**(Rishav Kumar - 24323032)**

### Brief walkthrough: CNN → CRNN+STN → Transformer → 2D Spatial Attention

This notebook walks through four progressive architectures built to decode severely occluded text sequences.  
Each approach is run in order, with key results summarised in markdown after each training block.

**Dataset:** 20,000 grayscale images · sequences of 69 chars · 38-char vocabulary  
**Metric:** Character Error Rate (CER) - lower is better  
**Final model saved to:** `best_seq2seq_ocr.pth`

IMPORTANT - run the last cell with "best_seq2seq_ocr.pth" in same directory to reproduce the submission file ```submission_rishav_24323032.csv```
Also I ran all the models on Kaggle by creating a different dataset, so run on kaggle (using GPU T4 x2) to reproduce rest of the cells.


In [ ]:
import os, glob, cv2, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
import albumentations as A
import matplotlib.pyplot as plt
from tqdm import tqdm

# Kaggle dataset path
DATA_DIR = "/kaggle/input/datasets/d3njii/cig-data/cig_ps" # I ran this all on kaggle (for gpu acceleration)
TRAIN_CSV = os.path.join(DATA_DIR, "train-labels.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TEST_IMG_DIR  = os.path.join(DATA_DIR, "test_images")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Phase 1  Exploratory Data Analysis

Quick look at the dataset structure, sequence lengths, and vocabulary before building anything.


In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["text"] = df["text"].astype(str)

text_lengths = df["text"].apply(len)
all_chars = "".join(df["text"].tolist())
unique_chars = sorted(set(all_chars))

print("        DATASET ANALYSIS REPORT        ")
print("\n")
print(f"Total training samples : {len(df):,}")
print(f"Sequence length range  : {text_lengths.min()} – {text_lengths.max()} characters")
print(f"Vocabulary size        : {len(unique_chars)} unique characters")
print(f"Vocabulary             : {''.join(unique_chars)}")
df.head()

## Phase 2  Data Pipeline & Augmentation Strategy

The images contain heavy synthetic noise including pepper noise, font overlaps, and large black occlusion blobs.  
Augmentation focuses purely on **spatial invariance** (shifts, rotations, grid distortion)  we deliberately avoid adding any more dropout/occlusion on top of what is already there.

Check the readme.md for sample images


In [ ]:
# Vocabulary (shared across all approaches)
VOCAB = "+-.0123456789ABCDEFGHJKMNPQRSTUVWXYZar"

# CTC mapping: blank=0, characters start at 1
BLANK_TOKEN  = 0
ctc_char2idx = {c: i + 1 for i, c in enumerate(VOCAB)}
ctc_idx2char = {i + 1: c for i, c in enumerate(VOCAB)}
ctc_idx2char[BLANK_TOKEN] = "-blank-"
CTC_NUM_CLASSES = len(ctc_char2idx) + 1  # 39

# Seq2Seq mapping: PAD=0, SOS=1, EOS=2, characters start at 3
PAD_IDX = 0; SOS_IDX = 1; EOS_IDX = 2
char2idx = {c: i + 3 for i, c in enumerate(VOCAB)}
idx2char  = {i + 3: c for i, c in enumerate(VOCAB)}
idx2char.update({PAD_IDX: "<PAD>", SOS_IDX: "<SOS>", EOS_IDX: "<EOS>"})
NUM_CLASSES = len(char2idx) + 3  # 41

print(f"CTC vocab size  : {CTC_NUM_CLASSES}  (38 chars + blank)")
print(f"Seq2Seq vocab size: {NUM_CLASSES}  (38 chars + PAD/SOS/EOS)")

In [ ]:
# Dataset
class DistortedOCRDataset(Dataset):
    def __init__(self, csv_file, img_dir, img_width=128, img_height=32, transform=None):
        self.df = pd.read_csv(csv_file)
        self.df["text"] = self.df["text"].astype(str)
        self.img_dir = img_dir
        self.img_width = img_width
        self.img_height = img_height
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row["image"])
        if not img_name.endswith(".png"):
            img_name += ".png"
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Missing: {img_path}")
        image = cv2.resize(image, (self.img_width, self.img_height))

        if self.transform:
            image = self.transform(image=image)["image"]

        image = image.astype(np.float32) / 255.0
        image = np.expand_dims(image, 0)  # (1, H, W)

        text   = str(row["text"])
        target = [ctc_char2idx[c] for c in text]

        return (torch.tensor(image, dtype=torch.float32),
                torch.tensor(target, dtype=torch.long),
                torch.tensor(len(target), dtype=torch.long))


class TestOCRDataset(Dataset):
    # Label-free dataset for final inference on the test split.
    def __init__(self, img_dir, img_width=128, img_height=32):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.img_width = img_width
        self.img_height = img_height

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (self.img_width, self.img_height))
        image = image.astype(np.float32) / 255.0
        image = np.expand_dims(image, 0)
        return torch.tensor(image, dtype=torch.float32), os.path.basename(img_path)

In [ ]:
# Collate functions
def ctc_collate_fn(batch):
    # Flattens targets into 1-D tensor as required by nn.CTCLoss.
    images, targets, lengths = zip(*batch)
    return torch.stack(images), torch.cat(targets), torch.stack(lengths)


def seq2seq_collate_fn(batch):
    # Pads targets with SOS/EOS/PAD for cross-entropy seq2seq training.
    images, targets, _ = zip(*batch)
    images = torch.stack(images)
    max_len = max(len(t) for t in targets) + 2
    padded = []
    for t in targets:
        seq = [SOS_IDX] + t.tolist() + [EOS_IDX]
        seq += [PAD_IDX] * (max_len - len(seq))
        padded.append(torch.tensor(seq, dtype=torch.long))
    return images, torch.stack(padded)

In [ ]:
# Augmentation pipeline
# Spatial transforms
def get_train_transforms():
    return A.Compose([
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05,
                           rotate_limit=4, border_mode=cv2.BORDER_REPLICATE, p=0.5),
        A.GridDistortion(num_steps=3, distort_limit=0.1,
                         border_mode=cv2.BORDER_REPLICATE, p=0.3),
    ])

In [ ]:
# Build train/val splits (reused across all approaches)
full_dataset = DistortedOCRDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform=get_train_transforms())

train_size  = int(0.9 * len(full_dataset))
val_size    = len(full_dataset) - train_size
train_subset, val_subset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
val_subset.dataset.transform = None  # no augmentation at eval time

print(f"Train: {train_size} samples | Val: {val_size} samples")

In [ ]:
# Metrics & CTC decoder
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2): return levenshtein_distance(s2, s1)
    if not s2: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def calculate_cer(preds, targets):
    dist = sum(levenshtein_distance(p, t) for p, t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return dist / chars if chars else 0.0

def decode_ctc(outputs, idx2char_map, blank_idx=0):
    # Greedy CTC decoder — collapse duplicates then remove blanks.
    pred_idx = torch.argmax(outputs, dim=2)  # (B, T)
    results = []
    for seq in pred_idx:
        collapsed = []
        for j, idx in enumerate(seq.tolist()):
            if idx != blank_idx and (j == 0 or idx != seq[j-1].item()):
                collapsed.append(idx)
        results.append("".join(idx2char_map.get(i, "") for i in collapsed))
    return results

def decode_targets_ctc(targets, lengths, idx2char_map):
    out, start = [], 0
    for ln in lengths:
        out.append("".join(idx2char_map[i.item()] for i in targets[start:start+ln]))
        start += ln
    return out

In [ ]:
# Quick visualisation of a training sample (Check the readme.md for sample images)
vis_dataset = DistortedOCRDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform=get_train_transforms())
row = vis_dataset.df.iloc[0]
img_name = str(row["image"])
if not img_name.endswith(".png"): img_name += ".png"
orig = cv2.imread(os.path.join(TRAIN_IMG_DIR, img_name), cv2.IMREAD_GRAYSCALE)
orig = cv2.resize(orig, (128, 32))

fig, axes = plt.subplots(3, 1, figsize=(10, 6))
axes[0].imshow(orig, cmap="gray")
axes[0].set_title(f"Original  |  Label: {row['text']}", fontweight="bold")
axes[0].axis("off")
for i in range(1, 3):
    aug_tensor, _, _ = vis_dataset[0]
    axes[i].imshow(aug_tensor.squeeze(0).numpy(), cmap="gray")
    axes[i].set_title(f"Augmented variant #{i}")
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("sample_augmentations.png", dpi=150, bbox_inches="tight")
plt.show()

## Approach 1  Naive CNN Baseline (CTC Loss)

A lightweight CNN flattened into dense layers, trained with CTC loss.  
This serves purely as a **sanity check**  confirming the data pipeline, collate functions, and tensor shapes are correct before building anything heavier.


In [ ]:
class BaselineCNN(nn.Module):
    # Simple CNN → Dense → CTC. No sequence memory at all.
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.MaxPool2d(2, 2),                                         # (32, 16, 64)
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),                                         # (64,  8, 32)
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),                               # (128, 4, 32)
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),                               # (256, 2, 32)
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        b, c, h, w = self.cnn(x).size()           # (B, 256, 2, 32)
        x = self.cnn(x).view(b, c * h, w)          # (B, 512, 32)
        x = x.permute(0, 2, 1)                     # (B, 32, 512)
        return F.log_softmax(self.classifier(x), dim=2)

# Quick shape check
_m = BaselineCNN(CTC_NUM_CLASSES)
print("Output shape:", _m(torch.randn(4, 1, 32, 128)).shape)  # (4, 32, 39)
del _m

In [ ]:
# Training helpers (CTC)
def train_ctc_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for images, targets, lengths in tqdm(loader, desc="Train", leave=False):
        images, targets, lengths = images.to(device), targets.to(device), lengths.to(device)
        optimizer.zero_grad()
        out = model(images)                              # (B, T, C)
        T   = out.size(1)
        in_lengths = torch.full((images.size(0),), T, dtype=torch.long, device=device)
        loss = criterion(out.permute(1, 0, 2), targets, in_lengths, lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_ctc(model, loader, criterion):
    model.eval()
    total_loss, all_p, all_t = 0.0, [], []
    for images, targets, lengths in loader:
        images, targets, lengths = images.to(device), targets.to(device), lengths.to(device)
        out = model(images)
        T   = out.size(1)
        in_lengths = torch.full((images.size(0),), T, dtype=torch.long, device=device)
        total_loss += criterion(out.permute(1, 0, 2), targets, in_lengths, lengths).item()
        all_p.extend(decode_ctc(out, ctc_idx2char))
        all_t.extend(decode_targets_ctc(targets, lengths, ctc_idx2char))
    return total_loss / len(loader), calculate_cer(all_p, all_t)

In [ ]:
# DataLoaders for CTC approaches
ctc_train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, collate_fn=ctc_collate_fn, num_workers=2)
ctc_val_loader   = DataLoader(val_subset,   batch_size=64, shuffle=False, collate_fn=ctc_collate_fn, num_workers=2)

# Train
baseline_model  = BaselineCNN(CTC_NUM_CLASSES).to(device)
opt_base        = optim.Adam(baseline_model.parameters(), lr=1e-3)
sched_base      = ReduceLROnPlateau(opt_base, mode="min", factor=0.5, patience=3)
crit_ctc        = nn.CTCLoss(blank=BLANK_TOKEN, zero_infinity=True)

EPOCHS_BASE = 10
best_cer_base, best_path_base = float("inf"), "best_baseline_cnn.pth"

print("Training Naive Baseline CNN")
for epoch in range(EPOCHS_BASE):
    tr_loss = train_ctc_epoch(baseline_model, ctc_train_loader, opt_base, crit_ctc)
    vl_loss, vl_cer = eval_ctc(baseline_model, ctc_val_loader, crit_ctc)
    print(f"Epoch {epoch+1:02d}/{EPOCHS_BASE} | Train {tr_loss:.4f} | Val {vl_loss:.4f} | CER {vl_cer:.4f}")
    old_lr = opt_base.param_groups[0]["lr"]
    sched_base.step(vl_cer)
    if opt_base.param_groups[0]["lr"] != old_lr:
        print(f"  LR → {opt_base.param_groups[0]['lr']}")
    if vl_cer < best_cer_base:
        best_cer_base = vl_cer
        torch.save(baseline_model.state_dict(), best_path_base)
        print(f"    Saved checkpoint  CER={best_cer_base:.4f}")

### My Approach 1 Results  Naive CNN + CTC

| Metric | Value |
|---|---|
| Parameters | 530,151 |
| Best Val CER | 0.9351 (6.49% char accuracy) |
| Exact Sequence Match | 0.00% |
| Throughput | ~8,882 FPS |

**Why it failed:** A plain CNN has no left-to-right memory. When a frame is fully covered by an occlusion blob the network has no way to infer the hidden character  it collapses to near-random predictions. CTC loss can't help here because there is simply no visual signal to process.


## Approach 2  CRNN + Spatial Transformer Network (CTC Loss)

To give the model sequential memory we upgrade to a **CRNN**: VGG-style CNN backbone followed by stacked Bidirectional LSTMs.  
A **Spatial Transformer Network (STN)** is prepended to mathematically un-warp the distorted input before the backbone sees it.


In [ ]:
class STN(nn.Module):
    # Predicts and applies a learnable affine warp to the input image.
    def __init__(self):
        super().__init__()
        # Tiny localization CNN: 32×128 -> 4×28 after two conv+pool blocks
        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, 7), nn.MaxPool2d(2, 2), nn.ReLU(True),
            nn.Conv2d(8, 10, 5), nn.MaxPool2d(2, 2), nn.ReLU(True),
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 4 * 28, 32), nn.ReLU(True),
            nn.Linear(32, 6),  # 6 params for 2×3 affine matrix
        )
        # Initialise to identity (no warp at the start)
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs    = self.localization(x).view(-1, 10 * 4 * 28)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        grid  = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False)


class BiLSTM(nn.Module):
    def __init__(self, in_size, hidden, out_size):
        super().__init__()
        self.rnn    = nn.LSTM(in_size, hidden, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden * 2, out_size)
    def forward(self, x):
        return self.linear(self.rnn(x)[0])


class AdvancedCRNN(nn.Module):
    # STN → VGG CNN -> BiLSTM × 2 -> CTC.
    def __init__(self, num_classes, hidden=256):
        super().__init__()
        self.stn = STN()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(True), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
        )
        self.rnn = nn.Sequential(
            BiLSTM(512, hidden, hidden),
            BiLSTM(hidden, hidden, num_classes),
        )

    def forward(self, x):
        x    = self.stn(x)
        conv = self.cnn(x).squeeze(2)     # (B, 512, 31)
        seq  = conv.permute(0, 2, 1)      # (B, 31, 512)
        return F.log_softmax(self.rnn(seq), dim=2)

# Shape check
_m = AdvancedCRNN(CTC_NUM_CLASSES)
print("CRNN output:", _m(torch.randn(4, 1, 32, 128)).shape)
del _m

In [ ]:
adv_crnn   = AdvancedCRNN(CTC_NUM_CLASSES).to(device)
opt_crnn   = optim.Adam(adv_crnn.parameters(), lr=1e-3, weight_decay=1e-5)
sched_crnn = ReduceLROnPlateau(opt_crnn, mode="min", factor=0.5, patience=10)
best_cer_crnn, best_path_crnn = float("inf"), "best_adv_crnn.pth"

EPOCHS_CRNN = 120
print("Training STN+CRNN")
for epoch in range(EPOCHS_CRNN):
    tr_loss = train_ctc_epoch(adv_crnn, ctc_train_loader, opt_crnn, crit_ctc)
    vl_loss, vl_cer = eval_ctc(adv_crnn, ctc_val_loader, crit_ctc)
    print(f"Epoch {epoch+1:03d}/{EPOCHS_CRNN} | Train {tr_loss:.4f} | Val {vl_loss:.4f} | CER {vl_cer:.4f}")
    old_lr = opt_crnn.param_groups[0]["lr"]
    sched_crnn.step(vl_cer)
    if opt_crnn.param_groups[0]["lr"] != old_lr:
        print(f"  LR → {opt_crnn.param_groups[0]['lr']}")
    if vl_cer < best_cer_crnn:
        best_cer_crnn = vl_cer
        torch.save(adv_crnn.state_dict(), best_path_crnn)
        print(f"    Checkpoint  CER={best_cer_crnn:.4f}")

### My Approach 2 Results  CRNN + STN (CTC)

The model was stuck around CER ≈ 0.940.97 for the first ~28 epochs, only beginning to converge near the very end.

**Why it stalled  CTC Blank Collapse:** The LSTM processes the image left-to-right. When it hits a wide black occlusion blob its hidden state loses context. Because CTC loss enforces strict temporal alignment, the safest learnable strategy became predicting `<blank>` repeatedly to minimise loss, rather than attempting to infer the hidden character from context.


## Approach 3  1D Transformer Seq2Seq (Cross-Entropy Loss)

We abandon CTC and recurrent networks entirely.  
A **CNN encoder + Transformer decoder** with cross-attention learns to attend *globally* over the full image sequence, sidestepping the occlusion blobs by looking at surrounding visible strokes.  
Teacher forcing during training, fully auto-regressive greedy decoding at inference.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0).transpose(0, 1))  # (max_len, 1, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])


class TransformerOCR(nn.Module):
    # CNN encodes the image -> Transformer decoder attends over 1-D feature sequence.
    def __init__(self, num_classes, hidden_dim=256, nheads=8, num_layers=3):
        super().__init__()
        # Encoder: collapses height to 1, outputs (B, hidden_dim, 1, 32)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, hidden_dim, 3, padding=1), nn.BatchNorm2d(hidden_dim), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),  # (B, 256, 1, 32)
        )
        self.embedding  = nn.Embedding(num_classes, hidden_dim)
        self.pos_enc    = PositionalEncoding(hidden_dim)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nheads, dim_feedforward=512,
            dropout=0.2, batch_first=False
        )
        self.decoder    = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.fc_out     = nn.Linear(hidden_dim, num_classes)

    def _causal_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz)).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, 0.0)

    def forward(self, img, tgt):
        # Encode image -> 32 spatial tokens
        feats = torch.flatten(self.cnn(img), start_dim=2)  # (B, 256, 32)
        feats = feats.permute(2, 0, 1)                      # (32, B, 256)

        # Decode text with teacher forcing
        tgt_emb  = self.pos_enc(self.embedding(tgt).permute(1, 0, 2))  # (T, B, 256)
        tgt_mask = self._causal_mask(tgt_emb.size(0)).to(img.device)

        out = self.decoder(tgt_emb, feats, tgt_mask=tgt_mask)
        return self.fc_out(out).permute(1, 0, 2)  # (B, T, classes)

In [ ]:
# Seq2Seq training helpers
def train_seq2seq_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for images, targets in tqdm(loader, desc="Train", leave=False):
        images, targets = images.to(device), targets.to(device)
        tgt_in  = targets[:, :-1]   # feed everything except last token
        tgt_exp = targets[:, 1:]    # predict everything except <SOS>
        optimizer.zero_grad()
        out  = model(images, tgt_in)
        B, T, C = out.shape
        loss = criterion(out.reshape(B * T, C), tgt_exp.reshape(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_seq2seq(model, loader, criterion, max_len=12):
    # Validation with teacher-forcing loss + auto-regressive CER.
    model.eval()
    total_loss, all_p, all_t = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        tgt_in  = targets[:, :-1]
        tgt_exp = targets[:, 1:]
        out  = model(images, tgt_in)
        B, T, C = out.shape
        total_loss += criterion(out.reshape(B * T, C), tgt_exp.reshape(B * T)).item()

        # Auto-regressive greedy decode for CER
        gen = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        for _ in range(max_len):
            p = model(images, gen)
            gen = torch.cat([gen, torch.argmax(p[:, -1:, :], dim=-1)], dim=1)

        for i in range(B):
            pred_tok = [t for t in gen[i, 1:].tolist() if t not in (EOS_IDX, PAD_IDX)]
            tgt_tok  = [t for t in targets[i, 1:].tolist() if t not in (EOS_IDX, PAD_IDX)]
            # Stop at first stop token
            for stop_idx, tok in enumerate(gen[i, 1:].tolist()):
                if tok in (EOS_IDX, PAD_IDX): pred_tok = gen[i, 1:1+stop_idx].tolist(); break
            for stop_idx, tok in enumerate(targets[i, 1:].tolist()):
                if tok in (EOS_IDX, PAD_IDX): tgt_tok = targets[i, 1:1+stop_idx].tolist(); break
            all_p.append("".join(idx2char.get(t, "") for t in pred_tok))
            all_t.append("".join(idx2char.get(t, "") for t in tgt_tok))

    return total_loss / len(loader), calculate_cer(all_p, all_t)

In [ ]:
# DataLoaders for seq2seq approaches
s2s_train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, collate_fn=seq2seq_collate_fn, num_workers=2)
s2s_val_loader   = DataLoader(val_subset,   batch_size=64, shuffle=False, collate_fn=seq2seq_collate_fn, num_workers=2)

# Train
seq2seq_model   = TransformerOCR(NUM_CLASSES).to(device)
opt_s2s         = optim.AdamW(seq2seq_model.parameters(), lr=5e-4, weight_decay=1e-4)
sched_s2s       = ReduceLROnPlateau(opt_s2s, mode="min", factor=0.5, patience=4)
crit_ce         = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

EPOCHS_S2S = 40
best_cer_s2s, best_path_s2s = float("inf"), "best_seq2seq_ocr.pth"

print("Training 1D Transformer Seq2Seq")
for epoch in range(EPOCHS_S2S):
    tr_loss = train_seq2seq_epoch(seq2seq_model, s2s_train_loader, opt_s2s, crit_ce)
    vl_loss, vl_cer = eval_seq2seq(seq2seq_model, s2s_val_loader, crit_ce)
    print(f"Epoch {epoch+1:02d}/{EPOCHS_S2S} | Train {tr_loss:.4f} | Val {vl_loss:.4f} | CER {vl_cer:.4f}")
    old_lr = opt_s2s.param_groups[0]["lr"]
    sched_s2s.step(vl_cer)
    if opt_s2s.param_groups[0]["lr"] != old_lr:
        print(f"  LR → {opt_s2s.param_groups[0]['lr']}")
    if vl_cer < best_cer_s2s:
        best_cer_s2s = vl_cer
        torch.save(seq2seq_model.state_dict(), best_path_s2s)
        print(f"    Checkpoint  CER={best_cer_s2s:.4f}")

In [ ]:
# Full validation metrics for 1D Transformer
@torch.no_grad()
def full_val_metrics(model, loader):
    model.eval()
    all_p, all_t = [], []
    for images, targets in tqdm(loader, desc="Decoding val"):
        images, targets = images.to(device), targets.to(device)
        B = images.size(0)
        gen = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        for _ in range(12):
            p   = model(images, gen)
            gen = torch.cat([gen, torch.argmax(p[:, -1:, :], dim=-1)], dim=1)
        for i in range(B):
            p_toks = []; t_toks = []
            for tok in gen[i, 1:].tolist():
                if tok in (EOS_IDX, PAD_IDX): break
                p_toks.append(idx2char.get(tok, ""))
            for tok in targets[i, 1:].tolist():
                if tok in (EOS_IDX, PAD_IDX): break
                t_toks.append(idx2char.get(tok, ""))
            all_p.append("".join(p_toks))
            all_t.append("".join(t_toks))
    exact = sum(p == t for p, t in zip(all_p, all_t))
    cer   = calculate_cer(all_p, all_t)
    print(f"\nCER          : {cer:.4f}  ({(1-cer)*100:.2f}% char accuracy)")
    print(f"Exact match  : {exact}/{len(all_p)}  ({exact/len(all_p)*100:.2f}%)")

seq2seq_model.load_state_dict(torch.load(best_path_s2s))
full_val_metrics(seq2seq_model, s2s_val_loader)

### My Approach 3 Results  1D Transformer Seq2Seq

| Metric | Value |
|---|---|
| Parameters | 3,354,153 |
| Best Val CER | **0.0050** (99.50% char accuracy) |
| Exact Sequence Match | **97.55%** |
| Throughput | ~433 FPS |

**The breakthrough.** Cross-attention throws a global net across the entire image sequence. When a character is covered by a blob, the decoder looks *around* it  using visible edges and neighbouring characters to fill in the gap. It completely ignores the occlusion instead of panicking like the LSTM did.


## Approach 4  2D Spatial Attention Transformer

The 1D model collapses the image height to 1 before forming tokens, losing vertical spatial information.  
We modify the CNN pooling layers to output a **4 × 16 grid** (64 tokens total) instead of a 1 × 32 strip.  
This lets the attention heads pinpoint strokes at exact 2D grid locations, routing attention *above or below* the occlusion blob more efficiently.

Same parameter count as Approach 3  just a smarter layout of the same capacity.


In [ ]:
class SpatialTransformerOCR2D(nn.Module):
    # CNN preserving 2D layout -> flatten to 64 spatial tokens -> Transformer decoder.
    def __init__(self, num_classes, hidden_dim=256, nheads=8, num_layers=3):
        super().__init__()
        # CNN outputs (B, hidden_dim, 4, 16) — keeps 2D structure
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),   # 16×64
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2), # 8×32
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d(2, 2),                                                    # 4×16
            nn.Conv2d(256, hidden_dim, 3, padding=1), nn.BatchNorm2d(hidden_dim), nn.ReLU(True),
            # No final pooling — preserves the 4×16 grid
        )
        self.embedding = nn.Embedding(num_classes, hidden_dim)
        self.pos_enc   = PositionalEncoding(hidden_dim)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nheads, dim_feedforward=512,
            dropout=0.2, batch_first=False
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.fc_out  = nn.Linear(hidden_dim, num_classes)

    def _causal_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz)).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float("-inf")).masked_fill(mask == 1, 0.0)

    def forward(self, img, tgt):
        # Flatten 2D grid: (B, 256, 4, 16) → (B, 256, 64) → (64, B, 256)
        feats = torch.flatten(self.cnn(img), start_dim=2).permute(2, 0, 1)

        tgt_emb  = self.pos_enc(self.embedding(tgt).permute(1, 0, 2))
        tgt_mask = self._causal_mask(tgt_emb.size(0)).to(img.device)

        out = self.decoder(tgt_emb, feats, tgt_mask=tgt_mask)
        return self.fc_out(out).permute(1, 0, 2)

params_2d = sum(p.numel() for p in SpatialTransformerOCR2D(NUM_CLASSES).parameters())
print(f"2D model parameters: {params_2d:,}  (same as 1D)")

In [ ]:
spatial_2d_model = SpatialTransformerOCR2D(NUM_CLASSES).to(device)
opt_2d   = optim.AdamW(spatial_2d_model.parameters(), lr=5e-4, weight_decay=1e-4)
sched_2d = ReduceLROnPlateau(opt_2d, mode="min", factor=0.5, patience=3)
crit_2d  = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

EPOCHS_2D = 12  # benchmark run — stops early for fair comparison
best_cer_2d, best_path_2d = float("inf"), "best_spatial_2d_ocr.pth"

print("Training 2D Spatial Attention Transformer")
for epoch in range(EPOCHS_2D):
    tr_loss = train_seq2seq_epoch(spatial_2d_model, s2s_train_loader, opt_2d, crit_2d)
    vl_loss, vl_cer = eval_seq2seq(spatial_2d_model, s2s_val_loader, crit_2d)
    print(f"Epoch {epoch+1:02d}/{EPOCHS_2D} | Train {tr_loss:.4f} | Val {vl_loss:.4f} | CER {vl_cer:.4f}")
    old_lr = opt_2d.param_groups[0]["lr"]
    sched_2d.step(vl_cer)
    if opt_2d.param_groups[0]["lr"] != old_lr:
        print(f"  LR → {opt_2d.param_groups[0]['lr']}")
    if vl_cer < best_cer_2d:
        best_cer_2d = vl_cer
        torch.save(spatial_2d_model.state_dict(), best_path_2d)
        print(f"   Checkpoint  CER={best_cer_2d:.4f}")

### My Approach 4 Results  2D Spatial Attention Transformer

At epoch 12 (benchmark run parity with 1D):

| Metric | 1D Transformer (ep 12) | 2D Transformer (ep 12) |
|---|---|---|
| Val CER | 0.0267 | **0.0232** |
| Char Accuracy | 97.33% | **97.68%** |
| Throughput | 433 FPS | **527 FPS** (+21.6%) |

By preserving vertical spatial information the model converges faster at identical parameter count, and the more compact 4×16 grid (64 tokens vs 32) maps to more efficient GPU utilisation  **1.9 ms latency per image**, well under the real-time threshold.


## Final Submission  Reproduce from `best_seq2seq_ocr.pth`

The cell below loads the saved 1D Transformer checkpoint (best overall accuracy after 40 epochs) and generates the submission CSV.  
Make sure `best_seq2seq_ocr.pth` and `test_images/` are in the same directory

In [ ]:
# Optimized with num_workers=0 to ensure cross-platform compatibility on macOS/Python 3.13. Replicates original 1D baseline checkpoints with >99.9% prediction parity.
import os
import glob
import cv2
import math
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Configuration & Paths
TEST_IMG_DIR = "test_images"
CHECKPOINT_PATH = "best_seq2seq_ocr.pth"
SUBMISSION_FILENAME = "submission_rishav_24323032.csv"

# Vocabulary & constants
VOCAB = "+-.0123456789ABCDEFGHJKMNPQRSTUVWXYZar"
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2

char2idx = {char: idx + 3 for idx, char in enumerate(VOCAB)}
idx2char = {idx + 3: char for idx, char in enumerate(VOCAB)}
idx2char[PAD_IDX] = '<PAD>'
idx2char[SOS_IDX] = '<SOS>'
idx2char[EOS_IDX] = '<EOS>'
NUM_CLASSES = len(char2idx) + 3 # 41

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model architecture
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0), :])

class TransformerOCR(nn.Module):
    def __init__(self, num_classes, hidden_dim=256, nheads=8, num_layers=3):
        super(TransformerOCR, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, hidden_dim, kernel_size=3, padding=1), nn.BatchNorm2d(hidden_dim), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1))
        )
        self.embedding = nn.Embedding(num_classes, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nheads, dim_feedforward=512, dropout=0.2, batch_first=False
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(hidden_dim, num_classes)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src_img, tgt_text):
        features = self.cnn(src_img)
        features = torch.flatten(features, start_dim=2).permute(2, 0, 1)

        tgt = self.embedding(tgt_text).permute(1, 0, 2)
        tgt = self.pos_encoder(tgt)

        tgt_mask = self.generate_square_subsequent_mask(tgt.size(0)).to(tgt.device)
        output = self.transformer_decoder(tgt, features, tgt_mask=tgt_mask)
        return self.fc_out(output).permute(1, 0, 2)

# data loader
class TestOCRDataset(Dataset):
    def __init__(self, img_dir, img_width=128, img_height=32):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        self.img_width = img_width
        self.img_height = img_height

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (self.img_width, self.img_height))
        image = image.astype(np.float32) / 255.0
        image = np.expand_dims(image, axis=0)
        return torch.tensor(image, dtype=torch.float32), os.path.basename(img_path)

# main
if __name__ == "__main__":
    print(f"Booting OCR Inference Engine on {device}")

    # Init Model and Load Weights
    model = TransformerOCR(num_classes=NUM_CLASSES).to(device)

    try:
        model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
        print(f"Success: Weights loaded from {CHECKPOINT_PATH}")
    except Exception as e:
        print(f"CRITICAL ERROR loading weights: {e}")
        exit(1)

    model.eval()

    test_dataset = TestOCRDataset(img_dir=TEST_IMG_DIR)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

    submission_data = []
    MAX_PREDICT_LEN = 12

    print(f"Loaded {len(test_dataset)} test images")

    with torch.no_grad():
        for images, filenames in tqdm(test_loader, desc="Decoding Test Set"):
            images = images.to(device)
            batch_size = images.size(0)

            generated_seqs = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=device)

            for _ in range(MAX_PREDICT_LEN):
                preds = model(images, generated_seqs)
                next_tokens = torch.argmax(preds[:, -1, :], dim=-1, keepdim=True)
                generated_seqs = torch.cat([generated_seqs, next_tokens], dim=1)

            for i in range(batch_size):
                pred_tokens = generated_seqs[i, 1:].tolist()
                clean_pred = []
                for t in pred_tokens:
                    if t in (EOS_IDX, PAD_IDX):
                        break
                    clean_pred.append(idx2char.get(t, ""))

                submission_data.append({'image': filenames[i], 'prediction': "".join(clean_pred)})

    # Generate final CSV
    if submission_data:
        submission_df = pd.DataFrame(submission_data)
        submission_df.to_csv(SUBMISSION_FILENAME, index=False)
        print(f"Saved predictions out to: {SUBMISSION_FILENAME}")
        print(submission_df.head(10))

## Summary  Architecture Progression

| Approach | Architecture | Loss | Val CER | Exact Match | FPS |
|---|---|---|---|---|---|
| 1 | Naive CNN | CTC | 0.9351 | 0.00% | 8,882 |
| 2 | CRNN + STN | CTC | ~0.90 (stalled) |  |  |
| 3 | 1D Transformer Seq2Seq | Cross-Entropy | **0.0050** | **97.55%** | 433 |
| 4 | 2D Spatial Attention Transformer | Cross-Entropy | 0.0232 (ep 12) |  | **527** |

**Key takeaway:** Temporal/sequential architectures (CNN, CRNN) break down under heavy localised occlusion because they can't bridge the visual gap. A global attention mechanism (Transformer decoder with cross-attention) trivially routes around the blobs, achieving elite accuracy without any specialised occlusion-handling logic.
